# Data Cleaning

## Importing data from the World Bank API and Global Findex Database from the most recent year of publication (2024)

In [15]:
# Import necessary libraries
import pandas as pd
import requests
print("All libraries imported successfully.")

All libraries imported successfully.


## Data Collection

In [16]:
# Import the Global Findex Database 2025 dataset
global_findex = pd.read_csv("../data/raw/GlobalFindexDatabase2025.csv")

# See the shape of the dataset
print("The dataset has", global_findex.shape[0], "rows and", global_findex.shape[1], "columns.")

# See the first five rows of the dataset
global_findex.head()

The dataset has 8564 rows and 437 columns.


/var/folders/rn/1ww2hhjn0r58pz6lrgp_90sw0000gn/T/ipykernel_6344/420564996.py:2: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  global_findex = pd.read_csv("../data/raw/GlobalFindexDatabase2025.csv")


,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,fiaccount_t_d,...,con12m_s,con26lm_s,con12w_s,con2f_s,con13_s,con26m_s,con28lm_s,con5a_s,con17c_s,con32h_s
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Import gender, education and income data from the World Bank API

# Create a function to fetch World Bank indicator data for a given indicator code and year and return a cleaned DataFrame with country code, country name, and indicator value
def fetch_wb_indicator(indicator_code, value_col, year="2024"):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
    params = {"format": "json", "date": year, "per_page": 20000}
    response = requests.get(url, params=params)
    worldbankdata = response.json()[1]

    return pd.DataFrame([
        {"country_code" : entry["countryiso3code"],
         value_col : entry["value"]}
         for entry in worldbankdata
         if entry["value"] is not None and entry["countryiso3code"]
    ])

# Define the World Bank indicator codes for the relevant indicators
indicators = {
    "account_female": "FX.OWN.TOTL.FE.ZS",
    "account_male": "FX.OWN.TOTL.MA.ZS",
    "account_poor40": "FX.OWN.TOTL.40.ZS",
    "account_rich60": "FX.OWN.TOTL.60.ZS",
    "account_prim_less": "FX.OWN.TOTL.PL.ZS",
    "account_sec_more": "FX.OWN.TOTL.SO.ZS",}

# Fetch the data for each indicator and store in a dictionary of DataFrames
demographics = None
for col_name, code in indicators.items():
    df = fetch_wb_indicator(code, col_name)
    demographics = df if demographics is None else demographics.merge(df, on="country_code", how="outer")

# See the shape of the merged demographics DataFrame and save to csv
print(demographics.shape)
print(demographics.head())
demographics.to_csv("../data/raw/demographics.csv", index=False)

(147, 7)
  country_code  account_female  account_male  account_poor40  account_rich60  \
0          ALB       41.156049     51.450899       30.174878       56.657176   
1          ARG       84.249743     78.939553       74.664966       86.441764   
2          ARM       67.987980     75.566924       64.043162       76.237141   
3          AUS       97.724321     98.302651       96.817501       98.806206   
4          AUT       99.187327     99.837890       99.190969       99.719324   

   account_prim_less  account_sec_more  
0          26.201757         62.257660  
1          62.931393         88.299005  
2          57.452222         72.649618  
3          83.775932         98.490726  
4         100.000000         99.469281  


In [18]:
# Retrieve GDP per capita data for 2024 from the World Bank API
gdp_per_capita = fetch_wb_indicator("NY.GDP.PCAP.CD", "gdp_per_capita")
gdp_per_capita.to_csv("../data/raw/world_bank_gdp_per_capita.csv", index=False)
print(gdp_per_capita.shape)
print(gdp_per_capita.head())    

(236, 2)
  country_code  gdp_per_capita
0          AFE     1615.396356
1          AFW     1411.337029
2          ARB     7583.811701
3          CSS    19903.811231
4          CEB    24604.806860


## Data Cleaning

In [19]:
# First clean the Global Findex data

# Keep only data from the year 2024
globalfindex_24 = global_findex[global_findex["year"] == 2024]

# Now see the shape of the dataset 
print(globalfindex_24.shape)


(1969, 437)


In [20]:
# Keep the original data untouched 
findex_raw24 = globalfindex_24.copy()

# Create a list of the relevant columns to keep
cols_to_keep = ["countrynewwb", "codewb", "year", "pop_adult", "regionwb24_hi", "incomegroupwb24", "account_t_d", "fiaccount_t_d", "mobileaccount_t_d",]

# Create a cleaned dataset with only the relevant columns
findex_clean = globalfindex_24[cols_to_keep].copy()
print("We've kept", findex_clean.shape[1], "columns in the cleaned dataset.")
print("The columns in the cleaned dataset are:")
for col in findex_clean.columns:
    print("-", col)

#Reset the index
findex_clean.reset_index(drop=True)

We've kept 9 columns in the cleaned dataset.
The columns in the cleaned dataset are:
- countrynewwb
- codewb
- year
- pop_adult
- regionwb24_hi
- incomegroupwb24
- account_t_d
- fiaccount_t_d
- mobileaccount_t_d


,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,account_t_d,fiaccount_t_d,mobileaccount_t_d
0,Albania,ALB,2024,2278004.0,Europe & Central Asia (excluding high income),Upper middle income,0.460693,0.460693,NaN
1,Algeria,DZA,2024,32019804.0,Middle East & North Africa (excluding high inc...,Lower middle income,0.352901,0.352901,NaN
2,Argentina,ARG,2024,35433130.0,Latin America & Caribbean (excluding high income),Upper middle income,0.817442,0.728990,0.567395
3,Armenia,ARM,2024,2406318.0,Europe & Central Asia (excluding high income),Upper middle income,0.713735,0.705333,0.173603
4,Australia,AUS,2024,21852264.0,High income,High income,0.980104,0.980104,NaN
...,...,...,...,...,...,...,...,...,...
1964,Low income,LIC,2024,NaN,NaN,NaN,0.376106,0.226662,0.209943
1965,Lower middle income,LMC,2024,NaN,NaN,NaN,0.755855,0.688198,0.383805
1966,Lower middle income,LMC,2024,NaN,NaN,NaN,0.660206,0.617635,0.126635
1967,Upper middle income,UMC,2024,NaN,NaN,NaN,0.879884,0.869421,0.158765


In [21]:
# Rename columns in cleaned dataset for easier analysis
findex_clean = findex_clean.rename(columns={
    "countrynewwb": "country",
    "codewb": "country_code",
    "pop_adult": "adult_pop",
    "regionwb24_hi": "region",
    "incomegroupwb24": "income_group",
    "account_t_d": "percent_with_fin_account",
    "fiaccount_t_d": "percent_with_fin_account_formal",
    "mobileaccount_t_d": "percent_with_mobile_account",
})

print("Columns renamed successfully. Here are the new column names:")
for col in findex_clean.columns:
    print("-", col)

Columns renamed successfully. Here are the new column names:
- country
- country_code
- year
- adult_pop
- region
- income_group
- percent_with_fin_account
- percent_with_fin_account_formal
- percent_with_mobile_account


In [22]:
# Check shape of the cleaned dataset
print("The cleaned dataset has", findex_clean.shape[0], "rows and", findex_clean.shape[1], "columns.")

The cleaned dataset has 1969 rows and 9 columns.


In [23]:
# The cleaned dataset has too many rows, should have around 150 rows 
# Most likely there are multiple rows per country, so we need to keep only one row per country. We can do this by keeping the first row which should be the total for that country
findex_clean = findex_clean.drop_duplicates(subset=["country_code"], keep="first")
print("After dropping duplicates, the cleaned dataset has", findex_clean.shape[0], "rows and", findex_clean.shape[1], "columns.")

# Save the cleaned dataset to csv
findex_clean.to_csv("../data/interim/findex_clean.csv", index=False)

After dropping duplicates, the cleaned dataset has 152 rows and 9 columns.


In [24]:
# GDP per capita data from the World Bank also has too many rows, likely regions are included
#Filter out regional groups in GDP per capita data to keep only actual countries
url = "https://api.worldbank.org/v2/country?format=json&per_page=300"
countries = requests.get(url).json()[1]

valid_countries = {
    c['id'] for c in countries 
    if c['region']['value'] != 'Aggregates'
}

gdp_per_capita_countries = gdp_per_capita[gdp_per_capita['country_code'].isin(valid_countries)]
print(gdp_per_capita_countries.shape)
gdp_per_capita_countries.head(10)

#Save this cleaned data as a csv file
gdp_per_capita_countries.to_csv("../data/interim/worldbank_gdp_clean.csv", index=False)

(192, 2)


In [25]:
# Check country codes match between the three datasets
print(findex_clean['country_code'].head())
print(demographics['country_code'].head())
print(gdp_per_capita_countries['country_code'].head())


571    ALB
572    DZA
573    ARG
574    ARM
575    AUS
Name: country_code, dtype: object
0    ALB
1    ARG
2    ARM
3    AUS
4    AUT
Name: country_code, dtype: object
44    ALB
45    DZA
46    AND
47    AGO
48    ATG
Name: country_code, dtype: object


In [26]:
# Merge the three datasets on country code to create a master dataset for analysis
merged = findex_clean.merge(demographics, on="country_code", how="left").merge(gdp_per_capita_countries, on="country_code", how="left")

print("Our master dataset has", merged.shape[0], "rows and", merged.shape[1], "columns.")
print(merged.head())

Our master dataset has 152 rows and 16 columns.
     country country_code  year   adult_pop  \
0    Albania          ALB  2024   2278004.0   
1    Algeria          DZA  2024  32019804.0   
2  Argentina          ARG  2024  35433130.0   
3    Armenia          ARM  2024   2406318.0   
4  Australia          AUS  2024  21852264.0   

                                              region         income_group  \
0      Europe & Central Asia (excluding high income)  Upper middle income   
1  Middle East & North Africa (excluding high inc...  Lower middle income   
2  Latin America & Caribbean (excluding high income)  Upper middle income   
3      Europe & Central Asia (excluding high income)  Upper middle income   
4                                        High income          High income   

   percent_with_fin_account  percent_with_fin_account_formal  \
0                  0.460693                         0.460693   
1                  0.352901                         0.352901   
2             

In [27]:
# Region column conatins income group aswell as the region
# Create column with pure geographic region
# World Bank country metadata API returns all countries with regions
url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
response = requests.get(url)
data = response.json()
countries = pd.json_normalize(data[1])

region_mapping = (
    countries[countries["region.value"] != "Aggregates"]
    [["id", "region.value"]]
    .rename(columns={"id": "country_code", "region.value": "region_geo"})
)

merged = merged.merge(region_mapping, on="country_code", how="left")    

# Save the merged dataset to csv
merged.to_csv("../data/processed/merged_dataset.csv", index=False)

merged.head()

,country,country_code,year,adult_pop,region,income_group,percent_with_fin_account,percent_with_fin_account_formal,percent_with_mobile_account,account_female,account_male,account_poor40,account_rich60,account_prim_less,account_sec_more,gdp_per_capita,region_geo
0,Albania,ALB,2024,2278004.0,Europe & Central Asia (excluding high income),Upper middle income,0.460693,0.460693,NaN,41.156049,51.450899,30.174878,56.657176,26.201757,62.257660,11377.775743,Europe & Central Asia
1,Algeria,DZA,2024,32019804.0,Middle East & North Africa (excluding high inc...,Lower middle income,0.352901,0.352901,NaN,18.070989,51.862915,31.248964,37.976993,25.017129,38.838922,5752.990767,"Middle East, North Africa, Afghanistan & Pakistan"
2,Argentina,ARG,2024,35433130.0,Latin America & Caribbean (excluding high income),Upper middle income,0.817442,0.728990,0.567395,84.249743,78.939553,74.664966,86.441764,62.931393,88.299005,13969.783660,Latin America & Caribbean
3,Armenia,ARM,2024,2406318.0,Europe & Central Asia (excluding high income),Upper middle income,0.713735,0.705333,0.173603,67.987980,75.566924,64.043162,76.237141,57.452222,72.649618,8556.214070,Europe & Central Asia
4,Australia,AUS,2024,21852264.0,High income,High income,0.980104,0.980104,NaN,97.724321,98.302651,96.817501,98.806206,83.775932,98.490726,64603.985631,East Asia & Pacific
